# REFERENCE SOLUTION W1D2: A subject by time matrix: axis reductions, SD, and broadcasting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dc00086-glitch/internship-prep/blob/main/solutions/week01/w1d2_numpy_subject_matrix_solution.ipynb)

| | |
|---|---|
| Problem ID | `W1D2` |
| Week / day | 1 / 2 |
| Topic | numpy: 2D arrays, axis reductions, sample SD, broadcasting |
| Expected time | 20-30 min |

> This is the worked solution. Only open it after you have genuinely tried the problem notebook.


## Scenario

A single ascending dose study gave 8 healthy volunteers the same 50 mg oral
dose. Each subject was sampled at the same 9 timepoints, so the data manager
hands you one rectangular block: `conc` with shape `(8, 9)`, one **row per
subject** and one **column per timepoint**.

Before anybody fits a model, the pharmacometrician reports the mean profile,
how variable the subjects are, and then overlays the profiles on a common
scale so shapes can be compared without the amplitude getting in the way.

Two house rules that apply to everything in this repo:

- Variability is reported as **standard deviation (SD)**, never SEM. SEM
  shrinks as you add subjects, which hides exactly the between-subject
  variability a pharmacometrician is paid to see.
- SD means the **sample** SD, with `n - 1` in the denominator. Note that
  `np.std` defaults to `n`, so you have to say so explicitly.

## How numpy works in two dimensions

**A 2D array is a rectangle with a meaning attached to each direction.** Here
rows are subjects and columns are timepoints, so `conc[3, 5]` is subject 4 at
timepoint 6, counting from zero. `conc.shape` reports `(8, 9)`, always as
`(rows, columns)`.

**Reductions and the axis argument.** A reduction is anything that turns many
numbers into fewer: mean, max, sum, std. On a 2D array you have to say which
direction to collapse, and that is what `axis` is for. The rule worth
memorising:

> **The axis you name is the one that disappears.**

So `axis=0` destroys the row direction and leaves one value per column, and
`axis=1` destroys the column direction and leaves one value per row. People
get this backwards constantly, which is why the fix is always the same: print
`.shape` of what came out and ask whether that length is the number of
subjects or the number of timepoints.

**ddof, and why the default is wrong for you.** The SD of a sample divides by
`n - 1` rather than `n`, because you used the data itself to estimate the mean
and that uses up one degree of freedom. Dividing by `n` gives a number that is
systematically too small. numpy defaults to `ddof=0`, dividing by `n`, which
suits physics and image processing where you often have the whole population.
It does not suit you, who almost always has a sample of subjects drawn from a
population. So in numpy you must write `ddof=1` every single time. Pandas made
the opposite choice and already uses `n - 1`, which is a real inconsistency
between the two libraries and a classic source of quietly wrong numbers.

**Broadcasting.** When you combine two arrays of different shapes, numpy tries
to stretch the smaller one to fit, comparing shapes **from the right**. A
`(8, 9)` and a `(9,)` line up, because 9 matches 9, and the `(9,)` is reused
for every row. A `(8, 9)` and an `(8,)` do **not** line up, because it checks
9 against 8 and gives up. This is the error you will hit in part c, and it is
not a bug in your thinking: you want to divide each row by its own number, so
you have to say so by making that array `(8, 1)` instead of `(8,)`. Adding
`keepdims=True` to the reduction does it, and so does `.reshape(-1, 1)`.

## Your task

Three parts, worked one at a time.

| Part | What you work out | Variables to fill in |
|---|---|---|
| a | the mean profile, and each subject's own peak | `mean_by_time`, `cmax_by_subject` |
| b | how much the subjects differ | `sd_by_time`, `cv_by_time` |
| c | every subject rescaled to their own peak | `norm_conc` |

Two house rules apply throughout. Variability is reported as **SD**, never
SEM. And SD means the **sample** SD with `n - 1`, which numpy will not give
you unless you ask with `ddof=1`.

Below this you will find the **Setup** cell, then a **Toolkit** cell that
demonstrates every move on a tiny made up example, then **Part a**, **Part b**
and **Part c**, each with its own explanation and a cell to write in.

## Setup

Run this cell first. It builds the data with a fixed random seed, so your numbers match the checker and match mine.

In [ ]:
import numpy as np

PROBLEM_ID = "W1D2"
TOPIC = "numpy: 2D arrays, axis reductions, sample SD, broadcasting"

rng = np.random.default_rng(1102)

DOSE_MG = 50.0
N_SUBJECTS = 8
time_h = np.array([0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 24.0])
subject_id = np.arange(1, N_SUBJECTS + 1)

# Between subject variability in clearance and volume, then assay noise on top.
_V = 40.0 * rng.lognormal(0.0, 0.25, size=N_SUBJECTS)
_kel = 0.10 * rng.lognormal(0.0, 0.30, size=N_SUBJECTS)
_ka = 0.90 * rng.lognormal(0.0, 0.15, size=N_SUBJECTS)

_V, _kel, _ka = _V[:, None], _kel[:, None], _ka[:, None]
_t = time_h[None, :]
_true = (0.85 * DOSE_MG * _ka) / (_V * (_ka - _kel)) * (np.exp(-_kel * _t) - np.exp(-_ka * _t))
conc = np.round(_true * rng.lognormal(0.0, 0.10, size=_true.shape), 3)

print("conc shape:", conc.shape, " (rows = subjects, columns = timepoints)")
print("subject_id:", subject_id)
print("time_h    :", time_h)
print(conc)


## Toolkit for this problem

The one idea to get straight here is **axis**. In a 2D array, the axis you
name is the one that **disappears**.

- `axis=0` collapses the rows, so you are left with **one value per column**.
- `axis=1` collapses the columns, so you are left with **one value per row**.

In this problem rows are subjects and columns are timepoints, so "per
timepoint" means `axis=0`, and "per subject" means `axis=1`.

| What you want to do | How |
|---|---|
| make a 2D array | `np.array([[1, 2, 3], [4, 5, 6]])` |
| its size | `a.shape`, which reads `(rows, columns)` |
| mean of each column | `a.mean(axis=0)` |
| mean of each row | `a.mean(axis=1)` |
| biggest in each row | `a.max(axis=1)` |
| sample SD, with `n - 1` | `a.std(axis=0, ddof=1)` |
| population SD, with `n` | `a.std(axis=0)`, the default, which is **not** what we want |
| keep the collapsed axis | `a.max(axis=1, keepdims=True)` gives shape `(rows, 1)` |
| divide every row by its own number | `a / a.max(axis=1, keepdims=True)` |

Run the cell below, then carry on to **Part a**, which is the next section down.

In [ ]:
import numpy as np

# A tiny made up example: 2 rows, 3 columns. Nothing here is the answer.
demo = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 9.0]])

print("demo:")
print(demo)
print("shape:", demo.shape, " <- (rows, columns)")

print("")
print("--- 1. the axis you name is the one that disappears ---")
print("demo.mean(axis=0) :", demo.mean(axis=0), " length", demo.mean(axis=0).size,
      " <- rows gone, one value per COLUMN")
print("demo.mean(axis=1) :", demo.mean(axis=1), " length", demo.mean(axis=1).size,
      " <- columns gone, one value per ROW")
print("demo.max(axis=1)  :", demo.max(axis=1), " <- biggest in each row")

print("")
print("--- 2. sample SD versus population SD ---")
print("std(axis=0)          :", np.round(demo.std(axis=0), 4), " <- divides by n, the default")
print("std(axis=0, ddof=1)  :", np.round(demo.std(axis=0, ddof=1), 4), " <- divides by n-1, what we want")

print("")
print("--- 3. why plain max(axis=1) will not divide cleanly ---")
demo_rowmax = demo.max(axis=1)
print("demo_rowmax       :", demo_rowmax, " shape", demo_rowmax.shape)
try:
    demo / demo_rowmax
except ValueError as err:
    print("demo / demo_rowmax fails with:", err)

print("")
print("--- 4. keepdims makes the shapes line up ---")
demo_rowmax_col = demo.max(axis=1, keepdims=True)
print("with keepdims     :", demo_rowmax_col.tolist(), " shape", demo_rowmax_col.shape)
print("demo / that:")
print(demo / demo_rowmax_col, " <- every row now peaks at 1.0")
print("")
print("reshape(-1, 1) does the same thing:", demo_rowmax.reshape(-1, 1).shape)


## Part a. Reduce along the right axis

Both of these are one-line reductions. The only thing to get right is which
direction you collapse, and the rule is that the axis you name is the one that
disappears.

"Mean at each timepoint" means you want one value per column, so the rows have
to go. "Each subject's peak" means one value per row, so the columns have to
go. If you are unsure which you got, print `.shape` of the result and ask
whether that length is 8 or 9.

Fill in `mean_by_time` (length 9), `cmax_by_subject` (length 8).
Moves: `conc.mean(axis=...)`, `conc.max(axis=...)`.

In [ ]:
# Part a. Reduce along the right axis.

mean_by_time = conc.mean(axis=0)
cmax_by_subject = conc.max(axis=1)

print("mean_by_time   ", np.shape(mean_by_time), mean_by_time)
print("cmax_by_subject", np.shape(cmax_by_subject), cmax_by_subject)


## Part b. Describe the variability

Same reduction direction as the mean in part a, since you want the spread
across subjects at each timepoint.

Two things to be deliberate about. This is a **sample** of subjects, so the SD
divides by `n - 1`, and numpy will not do that unless you ask with `ddof=1`.
And it is SD, not SEM, so nothing here should involve a square root of the
number of subjects.

CV% is just the SD expressed as a percentage of the mean, which makes
variability comparable across timepoints with very different magnitudes.

Fill in `sd_by_time`, `cv_by_time`, both length 9.
Moves: `conc.std(axis=..., ddof=1)`, then elementwise division and `* 100`.

In [ ]:
# Part b. Describe the variability. SD, not SEM. Sample SD, so ddof=1.

sd_by_time = conc.std(axis=0, ddof=1)
cv_by_time = 100.0 * sd_by_time / mean_by_time

print("sd_by_time", np.shape(sd_by_time), np.round(sd_by_time, 3) if sd_by_time is not None else None)
print("cv_by_time", np.shape(cv_by_time), np.round(cv_by_time, 1) if cv_by_time is not None else None)


## Part c. Put every subject on a common scale

Normalising each profile to its own peak strips out the amplitude differences
so you can compare the **shape** of absorption and elimination between
subjects. It is a standard first look before any modelling.

You already have each subject's peak from part a, but dividing straight by it
will fail, because a `(8, 9)` and an `(8,)` do not line up: numpy matches
shapes from the right and tries 9 against 8. Read the error when it happens,
it is a useful one to recognise. Make the divisor `(8, 1)` and it works.

Fill in `norm_conc`, shape `(8, 9)`, every row peaking at exactly 1.0.
Moves: `conc.max(axis=1, keepdims=True)`, or `.reshape(-1, 1)`, then divide.

In [ ]:
# Part c. Normalise each subject's profile to that subject's own Cmax.
# Use broadcasting, not a for loop.

norm_conc = conc / conc.max(axis=1, keepdims=True)

if norm_conc is not None:
    print("norm_conc shape:", np.shape(norm_conc))
    print("max of each row:", np.round(np.max(norm_conc, axis=1), 6))


## Check your work

Run this cell. It prints PASS, FAIL or TODO for each part. Do not edit it.

In [ ]:
# Grading helpers. You do not need to read or edit this cell.
import base64 as _b64, json as _json, datetime as _dt, textwrap as _tw
import numpy as _np
import pandas as _pd

# Expected answers, encoded so that you do not read them by accident.
_EXP = _json.loads(_b64.b64decode("eyJjbWF4X2J5X3N1YmplY3QiOiBbMC43OCwgMC43NDUsIDEuMzg0LCAwLjk5NSwgMC45NzMsIDAuODA3LCAwLjc4MiwgMC40ODVdLCAiY21heF9ieV90aW1lIjogWzAuNTcxLCAwLjc2LCAxLjE1OCwgMS4wOSwgMS4zODQsIDEuMDUyLCAwLjg4NSwgMC42MjIsIDAuMjczXSwgImN2X2J5X3RpbWUiOiBbMjguNTUzMzMwMjY2OTI1MTYyLCAyNS4xNDIyNzgyODE4OTQ0MTYsIDI2LjQyOTA2OTQ5NjUwMTg0MiwgMjQuNDc5MTkwNDU3MTEzOTU1LCA0Mi41ODU5NTc0NDYwMDQ0NywgMzcuMjEzMjU5MzIyNzI3MjQsIDQwLjM0MzU3ODAwNDIwNzA1LCA0My4zNjg5Njk5NjI4Nzg1MiwgNzAuMDY2ODIxNjAxOTIyMTddLCAiY3ZfZnJhY3Rpb24iOiBbMC4yODU1MzMzMDI2NjkyNTE2LCAwLjI1MTQyMjc4MjgxODk0NDE1LCAwLjI2NDI5MDY5NDk2NTAxODQ1LCAwLjI0NDc5MTkwNDU3MTEzOTU4LCAwLjQyNTg1OTU3NDQ2MDA0NDcsIDAuMzcyMTMyNTkzMjI3MjcyNCwgMC40MDM0MzU3ODAwNDIwNzA1NCwgMC40MzM2ODk2OTk2Mjg3ODUyMywgMC43MDA2NjgyMTYwMTkyMjE3XSwgIm1lYW5fYnlfc3ViamVjdCI6IFswLjQ1OTc3Nzc3Nzc3Nzc3Nzg0LCAwLjQ3NCwgMC44NTksIDAuNjc4MTExMTExMTExMTExMSwgMC42OTY2NjY2NjY2NjY2NjY3LCAwLjUxNzc3Nzc3Nzc3Nzc3NzgsIDAuNDc4ODg4ODg4ODg4ODg4ODYsIDAuMjg2Nzc3Nzc3Nzc3Nzc3OF0sICJtZWFuX2J5X3RpbWUiOiBbMC4zOTcxMjUwMDAwMDAwMDAwNiwgMC41OTEyNDk5OTk5OTk5OTk5LCAwLjc5MDc1MDAwMDAwMDAwMDEsIDAuODIwNSwgMC43NTg4NzUsIDAuNjQ5NzUsIDAuNTQ4MjUwMDAwMDAwMDAwMSwgMC4zNDA2MjUsIDAuMTEwMjVdLCAibl9zdWJqZWN0cyI6IDgsICJuX3RpbWVzIjogOSwgIm5vcm1fYnlfb3ZlcmFsbCI6IFtbMC4zMzM4MTUwMjg5MDE3MzQxNSwgMC4zOTMwNjM1ODM4MTUwMjksIDAuNDg5ODg0MzkzMDYzNTgzOSwgMC41NjM1ODM4MTUwMjg5MDE4LCAwLjM5NzM5ODg0MzkzMDYzNTksIDAuMzc5MzM1MjYwMTE1NjA3LCAwLjI2NDQ1MDg2NzA1MjAyMzE0LCAwLjE0MDg5NTk1Mzc1NzIyNTQ0LCAwLjAyNzQ1NjY0NzM5ODg0MzkzXSwgWzAuMjUxNDQ1MDg2NzA1MjAyMywgMC40NDIxOTY1MzE3OTE5MDc1NCwgMC41MjE2NzYzMDA1NzgwMzQ3LCAwLjUzODI5NDc5NzY4Nzg2MTMsIDAuNDA2MDY5MzY0MTYxODQ5OCwgMC4zNDYwOTgyNjU4OTU5NTM3NywgMC4zMTIxMzg3MjgzMjM2OTk0MywgMC4xOTg2OTk0MjE5NjUzMTc5NSwgMC4wNjU3NTE0NDUwODY3MDUyXSwgWzAuMzcyMTA5ODI2NTg5NTk1NCwgMC41NDMzNTI2MDExNTYwNjk0LCAwLjgzNjcwNTIwMjMxMjEzODcsIDAuNzg3NTcyMjU0MzM1MjYwMiwgMS4wLCAwLjc2MDExNTYwNjkzNjQxNjIsIDAuNjM5NDUwODY3MDUyMDIzMSwgMC40NDk0MjE5NjUzMTc5MTkxLCAwLjE5NzI1NDMzNTI2MDExNTYzXSwgWzAuMjkzMzUyNjAxMTU2MDY5NCwgMC40OTcxMDk4MjY1ODk1OTUzNSwgMC42NjQwMTczNDEwNDA0NjI1LCAwLjcxODkzMDYzNTgzODE1MDMsIDAuNjU2MDY5MzY0MTYxODQ5OCwgMC41OTY4MjA4MDkyNDg1NTUsIDAuNTI4OTAxNzM0MTA0MDQ2MywgMC4zNTExNTYwNjkzNjQxNjE5LCAwLjEwMzMyMzY5OTQyMTk2NTMxXSwgWzAuNDEyNTcyMjU0MzM1MjYwMSwgMC41NDkxMzI5NDc5NzY4Nzg2LCAwLjY2ODM1MjYwMTE1NjA2OTQsIDAuNjg1NjkzNjQxNjE4NDk3MSwgMC43MDMwMzQ2ODIwODA5MjQ5LCAwLjYyMjgzMjM2OTk0MjE5NjYsIDAuNTQxMTg0OTcxMDk4MjY1OSwgMC4yNDc4MzIzNjk5NDIxOTY1NywgMC4wOTk3MTA5ODI2NTg5NTk1Nl0sIFswLjI1MjE2NzYzMDA1NzgwMzQ2LCAwLjQxMDQwNDYyNDI3NzQ1NjY2LCAwLjU4MzA5MjQ4NTU0OTEzMzEsIDAuNTczNjk5NDIxOTY1MzE4LCAwLjUxMzAwNTc4MDM0NjgyMDksIDAuMzY2MzI5NDc5NzY4Nzg2MTYsIDAuMzg5NDUwODY3MDUyMDIzMiwgMC4yMjI1NDMzNTI2MDExNTYwNywgMC4wNTYzNTgzODE1MDI4OTAxOF0sIFswLjIwNTIwMjMxMjEzODcyODMzLCAwLjM1NzY1ODk1OTUzNzU3MjI2LCAwLjQ1NjY0NzM5ODg0MzkzMDcsIDAuNTY1MDI4OTAxNzM0MTA0MSwgMC40NjM4NzI4MzIzNjk5NDIyNiwgMC40NDk0MjE5NjUzMTc5MTkxLCAwLjMyMzY5OTQyMTk2NTMxNzksIDAuMjI2ODc4NjEyNzE2NzYzMDIsIDAuMDY1NzUxNDQ1MDg2NzA1Ml0sIFswLjE3NDg1NTQ5MTMyOTQ3OTc4LCAwLjIyNDcxMDk4MjY1ODk1OTU2LCAwLjM1MDQzMzUyNjAxMTU2MDcsIDAuMzA5OTcxMDk4MjY1ODk2LCAwLjI0NzEwOTgyNjU4OTU5NTQsIDAuMjM0ODI2NTg5NTk1Mzc1NzUsIDAuMTY5Nzk3Njg3ODYxMjcxNjcsIDAuMTMxNTAyODkwMTczNDEwNCwgMC4wMjE2NzYzMDA1NzgwMzQ2OF1dLCAibm9ybV9ieV90aW1lIjogW1swLjgwOTEwNjgzMDEyMjU5MjEsIDAuNzE1Nzg5NDczNjg0MjEwNSwgMC41ODU0OTIyMjc5NzkyNzQ3LCAwLjcxNTU5NjMzMDI3NTIyOTMsIDAuMzk3Mzk4ODQzOTMwNjM1OSwgMC40OTkwNDk0Mjk2NTc3OTQ3LCAwLjQxMzU1OTMyMjAzMzg5ODMsIDAuMzEzNTA0ODIzMTUxMTI1NDMsIDAuMTM5MTk0MTM5MTk0MTM5MTddLCBbMC42MDk0NTcwOTI4MTk2MTQ3LCAwLjgwNTI2MzE1Nzg5NDczNjgsIDAuNjIzNDg4NzczNzQ3ODQxMiwgMC42ODM0ODYyMzg1MzIxMSwgMC40MDYwNjkzNjQxNjE4NDk4LCAwLjQ1NTMyMzE5MzkxNjM0OTgsIDAuNDg4MTM1NTkzMjIwMzM4OTUsIDAuNDQyMTIyMTg2NDk1MTc2ODcsIDAuMzMzMzMzMzMzMzMzMzMzM10sIFswLjkwMTkyNjQ0NDgzMzYyNTMsIDAuOTg5NDczNjg0MjEwNTI2MywgMS4wLCAxLjAsIDEuMCwgMS4wLCAxLjAsIDEuMCwgMS4wXSwgWzAuNzExMDMzMjc0OTU2MjE3MywgMC45MDUyNjMxNTc4OTQ3MzY4LCAwLjc5MzYwOTY3MTg0ODAxMzksIDAuOTEyODQ0MDM2Njk3MjQ3NywgMC42NTYwNjkzNjQxNjE4NDk4LCAwLjc4NTE3MTEwMjY2MTU5NjgsIDAuODI3MTE4NjQ0MDY3Nzk2NiwgMC43ODEzNTA0ODIzMTUxMTI1LCAwLjUyMzgwOTUyMzgwOTUyMzddLCBbMS4wLCAxLjAsIDAuNzk4NzkxMDE4OTk4MjczLCAwLjg3MDY0MjIwMTgzNDg2MjMsIDAuNzAzMDM0NjgyMDgwOTI0OSwgMC44MTkzOTE2MzQ5ODA5ODg1LCAwLjg0NjMyNzY4MzYxNTgxOTIsIDAuNTUxNDQ2OTQ1MzM3NjIwNiwgMC41MDU0OTQ1MDU0OTQ1MDU1XSwgWzAuNjExMjA4NDA2MzA0NzI4NSwgMC43NDczNjg0MjEwNTI2MzE1LCAwLjY5Njg5MTE5MTcwOTg0NDcsIDAuNzI4NDQwMzY2OTcyNDc3LCAwLjUxMzAwNTc4MDM0NjgyMDksIDAuNDgxOTM5MTYzNDk4MDk4ODQsIDAuNjA5MDM5NTQ4MDIyNTk4OSwgMC40OTUxNzY4NDg4NzQ1OTgwNiwgMC4yODU3MTQyODU3MTQyODU3XSwgWzAuNDk3MzczMDI5NzcyMzI5MiwgMC42NTEzMTU3ODk0NzM2ODQyLCAwLjU0NTc2ODU2NjQ5Mzk1NTEsIDAuNzE3NDMxMTkyNjYwNTUwNCwgMC40NjM4NzI4MzIzNjk5NDIyNiwgMC41OTEyNTQ3NTI4NTE3MTEsIDAuNTA2MjE0Njg5MjY1NTM2NywgMC41MDQ4MjMxNTExMjU0MDE5LCAwLjMzMzMzMzMzMzMzMzMzMzNdLCBbMC40MjM4MTc4NjMzOTc1NDgyLCAwLjQwOTIxMDUyNjMxNTc4OTQ3LCAwLjQxODgyNTU2MTMxMjYwNzk1LCAwLjM5MzU3Nzk4MTY1MTM3NjEsIDAuMjQ3MTA5ODI2NTg5NTk1NCwgMC4zMDg5MzUzNjEyMTY3MzAwNCwgMC4yNjU1MzY3MjMxNjM4NDE4LCAwLjI5MjYwNDUwMTYwNzcxNzA2LCAwLjEwOTg5MDEwOTg5MDEwOTg3XV0sICJub3JtX2NvbmMiOiBbWzAuNTkyMzA3NjkyMzA3NjkyMywgMC42OTc0MzU4OTc0MzU4OTc1LCAwLjg2OTIzMDc2OTIzMDc2OTMsIDEuMCwgMC43MDUxMjgyMDUxMjgyMDUyLCAwLjY3MzA3NjkyMzA3NjkyMzEsIDAuNDY5MjMwNzY5MjMwNzY5MjMsIDAuMjUsIDAuMDQ4NzE3OTQ4NzE3OTQ4NzJdLCBbMC40NjcxMTQwOTM5NTk3MzE1LCAwLjgyMTQ3NjUxMDA2NzExNDEsIDAuOTY5MTI3NTE2Nzc4NTIzNSwgMS4wLCAwLjc1NDM2MjQxNjEwNzM4MjYsIDAuNjQyOTUzMDIwMTM0MjI4MiwgMC41Nzk4NjU3NzE4MTIwODA1LCAwLjM2OTEyNzUxNjc3ODUyMzUsIDAuMTIyMTQ3NjUxMDA2NzExNF0sIFswLjM3MjEwOTgyNjU4OTU5NTQsIDAuNTQzMzUyNjAxMTU2MDY5NCwgMC44MzY3MDUyMDIzMTIxMzg3LCAwLjc4NzU3MjI1NDMzNTI2MDIsIDEuMCwgMC43NjAxMTU2MDY5MzY0MTYyLCAwLjYzOTQ1MDg2NzA1MjAyMzEsIDAuNDQ5NDIxOTY1MzE3OTE5MSwgMC4xOTcyNTQzMzUyNjAxMTU2M10sIFswLjQwODA0MDIwMTAwNTAyNTE0LCAwLjY5MTQ1NzI4NjQzMjE2MDgsIDAuOTIzNjE4MDkwNDUyMjYxNCwgMS4wLCAwLjkxMjU2MjgxNDA3MDM1MTgsIDAuODMwMTUwNzUzNzY4ODQ0MiwgMC43MzU2NzgzOTE5NTk3OTksIDAuNDg4NDQyMjExMDU1Mjc2MzUsIDAuMTQzNzE4NTkyOTY0ODI0MTJdLCBbMC41ODY4NDQ4MDk4NjYzOTI1LCAwLjc4MTA4OTQxNDE4MjkzOTQsIDAuOTUwNjY4MDM2OTk4OTcyMywgMC45NzUzMzQwMTg0OTk0ODYxLCAxLjAsIDAuODg1OTE5ODM1NTYwMTIzMywgMC43Njk3ODQxNzI2NjE4NzA1LCAwLjM1MjUxNzk4NTYxMTUxMDgsIDAuMTQxODI5MzkzNjI3OTU0OF0sIFswLjQzMjQ2NTkyMzE3MjI0MjgsIDAuNzAzODQxMzg3ODU2MjU3NiwgMS4wLCAwLjk4Mzg5MDk1NDE1MTE3NzIsIDAuODc5ODAxNzM0ODIwMzIyMSwgMC42MjgyNTI3ODgxMDQwODkxLCAwLjY2NzkwNTgyNDAzOTY1MywgMC4zODE2NjA0NzA4Nzk4MDE3LCAwLjA5NjY1NDI3NTA5MjkzNjhdLCBbMC4zNjMxNzEzNTU0OTg3MjEyLCAwLjYzMjk5MjMyNzM2NTcyODksIDAuODA4MTg0MTQzMjIyNTA2NCwgMS4wLCAwLjgyMDk3MTg2NzAwNzY3MjcsIDAuNzk1Mzk2NDE5NDM3MzQwMSwgMC41NzI4OTAwMjU1NzU0NDc1LCAwLjQwMTUzNDUyNjg1NDIxOTksIDAuMTE2MzY4Mjg2NDQ1MDEyNzhdLCBbMC40OTg5NjkwNzIxNjQ5NDg0NiwgMC42NDEyMzcxMTM0MDIwNjE5LCAxLjAsIDAuODg0NTM2MDgyNDc0MjI2OCwgMC43MDUxNTQ2MzkxNzUyNTc4LCAwLjY3MDEwMzA5Mjc4MzUwNTIsIDAuNDg0NTM2MDgyNDc0MjI2OCwgMC4zNzUyNTc3MzE5NTg3NjI5LCAwLjA2MTg1NTY3MDEwMzA5Mjc4Nl1dLCAic2RfYnlfc3ViamVjdCI6IFswLjIzMDg4MjQyNTU4NTkzNDIyLCAwLjIxMzgwMTMwOTYzMTE2MTk4LCAwLjM1MTAwOTYxNTI1MjkxNTgsIDAuMjgxMzk0NDkzNzQ2OTY1NywgMC4yOTEzODMzMzg1NzY1MjE5LCAwLjI0MTMyMDIxMTQyOTYzNjQ3LCAwLjIxNzcyODg3MDY0MjE2MTU0LCAwLjEzNTE5MjI0OTk0MjIzNzZdLCAic2RfYnlfdGltZSI6IFswLjExMzM5MjQxMjgyMjUyNjU2LCAwLjE0ODY1MzcyMDM0MTcwMDcsIDAuMjA4OTg3ODY3MDQzNTg4MzUsIDAuMjAwODUxNzU3NzAwNjIwMDMsIDAuMzIzMTc0MTg0NTY4MzY2NCwgMC4yNDE3OTMxNTI0NDk0MjAyMywgMC4yMjExODM2NjY0MDgwNjUyMiwgMC4xNDc3MjU1NTM5MzYwNTQ5NywgMC4wNzcyNDg2NzA4MTYxMTkxOV0sICJzZF9ieV90aW1lX3BvcCI6IFswLjEwNjA2ODg4OTc2MDM4MTY3LCAwLjEzOTA1MjgyMjY5Njk4ODA3LCAwLjE5NTQ5MDI0OTExNzQ0MzE3LCAwLjE4Nzg3OTYxNTcxMTc2MzcsIDAuMzAyMzAxNzY4NzI2MjE4MywgMC4yMjYxNzY3ODM3MzM0MzI3MiwgMC4yMDY4OTgzNzQ4MTIzNzAxNCwgMC4xMzgxODQ2MDI1MjUwMjgxLCAwLjA3MjI1OTUxNDk0NDQwMDI0XSwgInNlbV9ieV90aW1lIjogWzAuMDQwMDkwMjcyMDIwOTU2NDc2LCAwLjA1MjU1NzAyNjg1MTExMjYsIDAuMDczODg4MzY4OTg2MTE2OTUsIDAuMDcxMDExODE5OTQxNjcyODgsIDAuMTE0MjU5MzI4NzA2MzYyMzgsIDAuMDg1NDg2Nzg4ODcwNzI4ODYsIDAuMDc4MjAwMjM1MjAyNDIzMDQsIDAuMDUyMjI4ODcwNDcxMzYxNzY2LCAwLjAyNzMxMTUyOTQ4NTg2MjYxNV19").decode("utf-8"))

CHECK_RESULTS = []


class _Todo(Exception):
    """Raised when a part has not been attempted yet."""


def _need(name):
    g = globals()
    if name not in g:
        raise _Todo("`" + name + "` is not defined yet. Run the starter cell for this part.")
    val = g[name]
    if val is None:
        raise _Todo("`" + name + "` is still None. Fill in the # YOUR CODE blank.")
    return val


def _as_scalar(val):
    try:
        arr = _np.asarray(val, dtype=float)
    except Exception:
        return None
    if arr.size != 1:
        return None
    return float(arr.reshape(-1)[0])


def _as_array(val):
    try:
        return _np.asarray(val, dtype=float)
    except Exception:
        return None


def _num_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_scalar(a)
    if a is None:
        return False
    return bool(_np.isclose(a, float(b), rtol=rtol, atol=atol, equal_nan=True))


def _arr_close(a, b, rtol=1e-4, atol=1e-8):
    a = _as_array(a)
    if a is None:
        return False
    b = _np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return False
    return bool(_np.allclose(a, b, rtol=rtol, atol=atol, equal_nan=True))


def _cols_of(obj):
    try:
        return [str(c) for c in obj.columns]
    except Exception:
        return None


def check():
    """Test every part and print PASS, FAIL or TODO for each one."""
    global CHECK_RESULTS
    names = sorted(n for n in globals() if n.startswith("_part_"))
    CHECK_RESULTS = []
    for name in names:
        label = name[len("_part_"):]
        try:
            problem = globals()[name]()
            status = "FAIL" if problem else "PASS"
            msg = problem or ""
        except _Todo as exc:
            status, msg = "TODO", str(exc)
        except Exception as exc:
            status = "FAIL"
            msg = "could not check this part, it raised %s: %s" % (type(exc).__name__, exc)
        CHECK_RESULTS.append((label, status, msg))
    _print_results()


def _print_results():
    bar = "=" * 70
    print(bar)
    print("Problem %s  |  %s" % (PROBLEM_ID, TOPIC))
    print(bar)
    for label, status, msg in CHECK_RESULTS:
        print("Part %s: %s" % (label, status))
        if msg:
            for line in _tw.wrap(msg, width=64):
                print("    " + line)
    n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
    print("-" * 70)
    print("Result: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    if n_pass == len(CHECK_RESULTS):
        print("All parts pass. Run the last cell to build your tutor block.")
    print(bar)


def send_to_tutor():
    """Build the block you copy to your tutor, along with your code."""
    lines = []
    lines.append("===== SEND TO TUTOR =====")
    lines.append("Problem ID : %s" % PROBLEM_ID)
    lines.append("Topic      : %s" % TOPIC)
    lines.append("Date       : %s" % _dt.date.today().isoformat())
    lines.append("Check results:")
    if not CHECK_RESULTS:
        lines.append("  check() has not been run yet. Run the check cell above first.")
    else:
        for label, status, msg in CHECK_RESULTS:
            lines.append("  Part %s: %s" % (label, status))
            if msg:
                for line in _tw.wrap(msg, width=58):
                    lines.append("      " + line)
        n_pass = sum(1 for _, s, _ in CHECK_RESULTS if s == "PASS")
        lines.append("Summary: %d of %d parts passing" % (n_pass, len(CHECK_RESULTS)))
    lines.append("=========================")
    lines.append("Paste your code cells below this block.")
    return "\n".join(lines)



def _part_a():
    mean_by_time = _need("mean_by_time")
    cmax_by_subject = _need("cmax_by_subject")
    n_sub, n_time = _EXP["n_subjects"], _EXP["n_times"]

    arr = _as_array(mean_by_time)
    if arr is None:
        return "`mean_by_time` should be a numeric array."
    if arr.ndim == 0:
        return ("`mean_by_time` is a single number, so you averaged the whole matrix. "
                "Pass an axis argument so the reduction only collapses one direction.")
    if arr.shape == (n_sub,):
        return ("`mean_by_time` has one value per subject, not per timepoint. You collapsed "
                "the wrong axis: to keep one value per column you have to collapse the rows.")
    if arr.shape != (n_time,):
        return "`mean_by_time` has shape %s but it should be (%d,), one value per timepoint." % (arr.shape, n_time)
    if not _arr_close(arr, _EXP["mean_by_time"]):
        return "`mean_by_time` has the right shape but the wrong values. It is a plain mean of conc, nothing else."

    arr = _as_array(cmax_by_subject)
    if arr is None:
        return "`cmax_by_subject` should be a numeric array."
    if arr.ndim == 0:
        return ("`cmax_by_subject` is a single number, so you took the max of the whole "
                "matrix. Pass an axis argument.")
    if arr.shape == (n_time,):
        return ("`cmax_by_subject` has one value per timepoint, not per subject. You "
                "collapsed the wrong axis: to keep one value per row you have to collapse "
                "the columns.")
    if arr.shape != (n_sub,):
        return "`cmax_by_subject` has shape %s but it should be (%d,), one value per subject." % (arr.shape, n_sub)
    if _arr_close(arr, _EXP["mean_by_subject"]):
        return "`cmax_by_subject` is each subject's mean, not each subject's peak."
    if not _arr_close(arr, _EXP["cmax_by_subject"]):
        return "`cmax_by_subject` has the right shape but the wrong values."
    return None


def _part_b():
    sd_by_time = _need("sd_by_time")
    cv_by_time = _need("cv_by_time")
    n_sub, n_time = _EXP["n_subjects"], _EXP["n_times"]

    arr = _as_array(sd_by_time)
    if arr is None:
        return "`sd_by_time` should be a numeric array."
    if arr.shape == (n_sub,):
        return ("`sd_by_time` has one value per subject. The SD you want is across subjects "
                "at a fixed timepoint, so collapse the rows and keep the columns.")
    if arr.shape != (n_time,):
        return "`sd_by_time` has shape %s but it should be (%d,), one value per timepoint." % (arr.shape, n_time)
    if _arr_close(arr, _EXP["sem_by_time"]):
        return ("`sd_by_time` is the SEM, not the SD. You divided by the square root of the "
                "number of subjects somewhere. This repo always reports SD.")
    if _arr_close(arr, _EXP["sd_by_time_pop"]):
        return ("`sd_by_time` used the population SD with n in the denominator. np.std "
                "defaults to ddof=0, so pass ddof=1 to get the sample SD.")
    if not _arr_close(arr, _EXP["sd_by_time"]):
        return "`sd_by_time` has the right shape but the wrong values."

    arr = _as_array(cv_by_time)
    if arr is None:
        return "`cv_by_time` should be a numeric array."
    if arr.shape != (n_time,):
        return "`cv_by_time` has shape %s but it should be (%d,), one value per timepoint." % (arr.shape, n_time)
    if _arr_close(arr, _EXP["cv_fraction"]):
        return "`cv_by_time` is a fraction. The task asks for a percent, so multiply by 100."
    if not _arr_close(arr, _EXP["cv_by_time"]):
        return ("`cv_by_time` is not 100 * SD / mean at each timepoint. Check that you paired "
                "your SD and your mean at the same timepoints, and that both are per timepoint.")
    return None


def _part_c():
    norm_conc = _need("norm_conc")
    n_sub, n_time = _EXP["n_subjects"], _EXP["n_times"]

    arr = _as_array(norm_conc)
    if arr is None:
        return "`norm_conc` should be a numeric array."
    if arr.shape != (n_sub, n_time):
        if arr.shape == (n_time, n_sub):
            return ("`norm_conc` has shape %s, which is the transpose of what is wanted. "
                    "Keep subjects on the rows." % (arr.shape,))
        return ("`norm_conc` has shape %s but it should be (%d, %d), the same shape as conc. "
                "If you got an error about shapes not aligning, that is the (8,) against "
                "(8, 9) problem the hint warns about." % (arr.shape, n_sub, n_time))

    row_max = _np.max(arr, axis=1)
    n_at_one = int(_np.sum(_np.isclose(row_max, 1.0)))
    if _arr_close(arr, _EXP["norm_by_overall"]):
        return ("`norm_conc` was divided by the single largest value in the whole matrix, so "
                "only one subject peaks at 1.0. Each subject needs their own Cmax.")
    if _arr_close(arr, _EXP["norm_by_time"]):
        return ("`norm_conc` was divided by the peak at each timepoint, which is a reduction "
                "down the columns. You want the peak of each subject, which is across the rows.")
    if n_at_one != n_sub:
        return ("`norm_conc` has the right shape but only %d of %d rows peak at exactly 1.0. "
                "Divide each row by the maximum of that same row." % (n_at_one, n_sub))
    if not _arr_close(arr, _EXP["norm_conc"]):
        return "`norm_conc` has the right shape and every row peaks at 1.0, but the other values do not match."
    return None


check()


## Send to tutor

In [ ]:
# Run this last. Copy everything it prints, then paste your own code under it.
print(send_to_tutor())
